In [24]:
import networkx as nx
import numpy as np
import matplotlib.pyplot as plt
import ot
import pandas as pd
import random
from gwgraphs import *

In [25]:
np.random.seed(42)
random.seed(42)

In [26]:
def generate_erdos_renyi_graphs_randnodes(n_graphs, n_nodes_range, p_edge):
  """
  Generates a list of Erdos-Renyi graphs with random number of nodes and probability of edge addition.

  Args:
    n_graphs (int): Number of graphs to generate
    n_nodes_range (tuple): Tuple of (min_nodes, max_nodes) for random node count
    p_edge (float): Probability of edge addition

  Returns:
    graphs: the list of generated graphs with a label
  """
  graphs = []
  min_nodes, max_nodes = n_nodes_range
  for _ in range(n_graphs):
    n_nodes = random.randint(min_nodes, max_nodes)
    graph = nx.erdos_renyi_graph(n_nodes, p_edge)
    graphs.append((graph, 1))  # Label as 1
  return graphs


def generate_sbm_graphs_randnodes(n_graphs, n_nodes_range, n_blocks, prob_matrix):
  """
  Generates a list of Stochastic Block Model graphs with random number of nodes, block sizes, and probabilities of edge addition within blocks and between blocks.
  Args:
    n_graphs (int): Number of graphs to generate
    n_nodes_range (tuple): Tuple of (min_nodes, max_nodes) for random node count
    n_blocks (int): Number of blocks in each graph
    prob_matrix (list): List of probabilities of edge addition within blocks and between blocks for each graph

  Returns:
    graphs: the list of generated graphs with a label
  """
  graphs = []
  min_nodes, max_nodes = n_nodes_range
  for _ in range(n_graphs):
    n_nodes = random.randint(min_nodes, max_nodes)
    # Randomly distribute nodes across blocks
    block_sizes = []
    remaining_nodes = n_nodes
    for i in range(n_blocks - 1):
      if remaining_nodes > 0:
        block_size = random.randint(1, max(1, remaining_nodes - (n_blocks - i - 1)))
        block_sizes.append(block_size)
        remaining_nodes -= block_size
      else:
        block_sizes.append(0)
    block_sizes.append(remaining_nodes)  # Assign remaining nodes to last block
    
    graph = nx.stochastic_block_model(block_sizes, prob_matrix)
    graphs.append((graph, -1))  # Label as -1
  return graphs


def compute_averages(graphs):
  # Initialize variables
  total_nodes = 0
  class_nodes = {}
  class_counts = {}

  # Iterate through graphs to compute totals
  for graph, label in graphs:
    num_nodes = graph.number_of_nodes()
    total_nodes += num_nodes

    if label not in class_nodes:
      class_nodes[label] = 0
      class_counts[label] = 0

    class_nodes[label] += num_nodes
    class_counts[label] += 1
  # Calculate total average
  total_avg = round(total_nodes / len(graphs))

  # Calculate class-based averages
  class_averages = {
      label: round(class_nodes[label] / count)
      for label, count in class_counts.items()
  }

  return total_avg, class_averages

In [27]:
# ERSBM1-Rand
prob_matrix = [
    [0.15, 0.05],  # Probabilities related to block 1 connections
    [0.05, 0.15]  # Probabilities related to block 2 connections
]

n_graphs = 100
#n_nodes_range_er = (85,125)
#n_nodes_range_sbm = (90,105)
n_nodes_range = (85,115)
n_blocks = 2

graphs_er = generate_erdos_renyi_graphs_randnodes(n_graphs, n_nodes_range, 0.1)
graphs_sbm = generate_sbm_graphs_randnodes(n_graphs, n_nodes_range, n_blocks, prob_matrix)

full_graph_data = graphs_er + graphs_sbm
# Convert graphs to pairwise distance matrices
distance_matrices = [
    compute_distance_matrix(graph) for graph, label in full_graph_data
]
# Prepare labels
labels = [label for graph, label in full_graph_data]

In [28]:
# ERSBM2-Rand
prob_matrix5 = [
    [0.45, 0.05],  # Probabilities related to block 1 connections
    [0.05, 0.45]  # Probabilities related to block 2 connections
]

graphs_er5 = generate_erdos_renyi_graphs_randnodes(n_graphs, n_nodes_range, 0.25)
graphs_sbm5 = generate_sbm_graphs_randnodes(n_graphs, n_nodes_range, n_blocks, prob_matrix5)

full_graph_data5 = graphs_er5 + graphs_sbm5
# Convert graphs to pairwise distance matrices
distance_matrices5 = [
    compute_distance_matrix(graph) for graph, label in full_graph_data5
]
# Prepare labels
labels5 = [label for graph, label in full_graph_data5]

In [29]:
# ERSBM1-Rand Stats
total_avg, class_averages = compute_averages(full_graph_data)
print(f"Average number of nodes in entire dataset: {total_avg}")
print("Average number of nodes for each class:")
for class_label, avg_nodes in class_averages.items():
        print(f"  Class {class_label}: {avg_nodes}")

# ERSBM2-Rand Stats
total_avg, class_averages = compute_averages(full_graph_data5)
print(f"Average number of nodes in entire dataset: {total_avg}")
print("Average number of nodes for each class:")
for class_label, avg_nodes in class_averages.items():
        print(f"  Class {class_label}: {avg_nodes}")

Average number of nodes in entire dataset: 100
Average number of nodes for each class:
  Class 1: 99
  Class -1: 100
Average number of nodes in entire dataset: 99
Average number of nodes for each class:
  Class 1: 99
  Class -1: 99


In [30]:
# GW-NCC: ERSBM1-Rand
exp = gw_barycenter_class_alg(distance_matrices, labels, num_folds=5, num_nodes=102, log=True)

Fold 0:
  Train: index=[  0   1   2   3   4   5   6   7   8  10  11  12  13  14  17  19  20  21
  22  23  24  25  26  27  28  29  31  32  33  34  35  36  37  38  39  40
  41  42  43  44  46  47  48  49  50  51  52  53  54  57  58  59  61  62
  63  64  70  71  72  73  74  77  79  80  81  83  85  86  87  88  89  90
  91  92  94  96  97  98  99 100 101 102 103 105 106 107 108 109 110 111
 112 113 114 116 117 118 119 120 121 122 123 126 127 129 130 131 133 134
 136 138 139 140 141 142 143 144 145 146 147 149 151 153 154 155 156 157
 159 160 161 162 163 166 167 168 169 171 172 173 175 176 178 179 180 181
 183 184 185 187 188 189 190 191 192 193 194 195 196 197 198 199]
  Test:  index=[  9  15  16  18  30  45  55  56  60  65  66  67  68  69  75  76  78  82
  84  93  95 104 115 124 125 128 132 135 137 148 150 152 158 164 165 170
 174 177 182 186]
GW barycenter for Class 1 graphs: [[0.1180234  2.0710195  1.9277478  ... 1.95004326 1.99923791 2.23661987]
 [2.0710195  0.13869734 2.09604571 ... 2.

In [31]:
# GW-NCC: ERSBM2-Rand
exp = gw_barycenter_class_alg(distance_matrices5, labels5, num_folds=5, num_nodes=101, log=True)

Fold 0:
  Train: index=[  0   1   2   3   4   5   6   7   8  10  11  12  13  14  17  19  20  21
  22  23  24  25  26  27  28  29  31  32  33  34  35  36  37  38  39  40
  41  42  43  44  46  47  48  49  50  51  52  53  54  57  58  59  61  62
  63  64  70  71  72  73  74  77  79  80  81  83  85  86  87  88  89  90
  91  92  94  96  97  98  99 100 101 102 103 105 106 107 108 109 110 111
 112 113 114 116 117 118 119 120 121 122 123 126 127 129 130 131 133 134
 136 138 139 140 141 142 143 144 145 146 147 149 151 153 154 155 156 157
 159 160 161 162 163 166 167 168 169 171 172 173 175 176 178 179 180 181
 183 184 185 187 188 189 190 191 192 193 194 195 196 197 198 199]
  Test:  index=[  9  15  16  18  30  45  55  56  60  65  66  67  68  69  75  76  78  82
  84  93  95 104 115 124 125 128 132 135 137 148 150 152 158 164 165 170
 174 177 182 186]
GW barycenter for Class 1 graphs: [[0.10038862 2.01965007 1.52142416 ... 1.4407192  1.54612269 1.51514198]
 [2.01965007 0.07407467 1.87922155 ... 1.